# 59. Tkinter Desktop Apps — `tkinter`
**Python Libraries Lab · Phase 3 · GUI, Games & Multimedia**

_Python's built-in toolkit for simple desktop windows._

💻 **Read and test in Colab, run the app on your laptop** (it opens a real window).

Tkinter is the desktop window toolkit that ships with Python, so there is nothing to install. You create a window, place widgets like labels, text boxes, drop downs and buttons, and connect buttons to Python functions. It is perfect for small tools for yourself or your team, like a price calculator or a file renamer.

**In this notebook:** You will build the ShopEasy order calculator. Pick a product, choose a quantity, tick a coupon box, and see the total update, then add the order to a list and save the list to a CSV file. The logic is tested here in Colab, and the window runs on your laptop.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
This library opens real windows, so it needs a computer with a screen. **In Colab** you read the code, check it, and (where possible) preview it without a window. **On your laptop** you run the real app: download the script from the lesson page or from Colab's Files panel, then follow the steps in the last cell. Tkinter comes with Python on Windows and Mac. On Linux: `sudo apt install python3-tk`.

## Step 1 · Logic first, testable anywhere
Keep calculations out of the window code so you can test them without a screen.

In [ ]:
%%writefile shop_logic.py
PRODUCTS = {"Steel Bottle": 499, "Yoga Mat": 1299, "Desk Lamp": 899, "Notebook": 149}
COUPONS = {"SAVE10": 0.10, "FEST25": 0.25}

def order_total(product, qty, coupon=""):
    if product not in PRODUCTS:
        raise ValueError("Unknown product")
    if qty < 1:
        raise ValueError("Quantity must be at least 1")
    subtotal = PRODUCTS[product] * qty
    discount = COUPONS.get(coupon.strip().upper(), 0)
    return round(subtotal * (1 - discount), 2)

## Step 2 · Test the logic in Colab
These checks run here; the window itself needs your laptop.

In [ ]:
import importlib, shop_logic
importlib.reload(shop_logic)
from shop_logic import order_total, PRODUCTS
print(order_total("Yoga Mat", 2))
print(order_total("Notebook", 10, "save10"))
try:
    order_total("Notebook", 0)
except ValueError as e:
    print("Error caught:", e)

## Step 3 · The Tkinter app
Widgets laid out with `grid`. `StringVar`s hold values; `trace_add` updates the total whenever they change.

In [ ]:
%%writefile tk_shop.py
import csv
import tkinter as tk
from tkinter import ttk, messagebox
from shop_logic import PRODUCTS, order_total

root = tk.Tk()
root.title("ShopEasy order calculator")
root.geometry("420x380")
frm = ttk.Frame(root, padding=16)
frm.pack(fill="both", expand=True)

product = tk.StringVar(value="Steel Bottle")
qty = tk.IntVar(value=1)
coupon = tk.StringVar()
total = tk.StringVar()

ttk.Label(frm, text="Product").grid(row=0, column=0, sticky="w", pady=4)
ttk.Combobox(frm, textvariable=product, values=list(PRODUCTS), state="readonly").grid(row=0, column=1, sticky="ew")
ttk.Label(frm, text="Quantity").grid(row=1, column=0, sticky="w", pady=4)
ttk.Spinbox(frm, from_=1, to=20, textvariable=qty, width=6).grid(row=1, column=1, sticky="w")
ttk.Label(frm, text="Coupon").grid(row=2, column=0, sticky="w", pady=4)
ttk.Entry(frm, textvariable=coupon).grid(row=2, column=1, sticky="ew")
ttk.Label(frm, textvariable=total, font=("Segoe UI", 14, "bold")).grid(row=3, column=0, columnspan=2, pady=10)
orders = tk.Listbox(frm, height=6)
orders.grid(row=5, column=0, columnspan=2, sticky="nsew", pady=6)
frm.columnconfigure(1, weight=1); frm.rowconfigure(5, weight=1)

def refresh(*_):
    try:
        total.set(f"Total: Rs. {order_total(product.get(), qty.get(), coupon.get()):,.2f}")
    except (ValueError, tk.TclError):
        total.set("Total: -")

def add_order():
    refresh()
    orders.insert("end", f"{qty.get()} x {product.get()}  ->  {total.get().split(': ')[1]}")

def save_csv():
    with open("orders.csv", "w", newline="") as f:
        csv.writer(f).writerows([[line] for line in orders.get(0, "end")])
    messagebox.showinfo("Saved", f"Saved {orders.size()} orders to orders.csv")

for var in (product, qty, coupon):
    var.trace_add("write", refresh)
ttk.Button(frm, text="Add to list", command=add_order).grid(row=4, column=0, pady=4, sticky="w")
ttk.Button(frm, text="Save CSV", command=save_csv).grid(row=4, column=1, pady=4, sticky="e")
refresh()
root.mainloop()

## Step 4 · Check the app file
Confirms the script is valid Python. Then follow the laptop steps below.

In [ ]:
import py_compile
for f in ["shop_logic.py", "tk_shop.py"]:
    py_compile.compile(f, doraise=True)
print("Both files are valid. Download shop_logic.py and tk_shop.py from the Files panel (left), put them in one folder, then run:  python tk_shop.py")

## ✍️ Your turn
Using `order_total`, store the total for **3 Desk Lamps with coupon `FEST25`** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 2022.75, f"Expected 2022.75, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = order_total("Desk Lamp", 3, "FEST25")

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Window (Tk)**
- **Widget**
- **grid / pack**
- **Event / command**
- **StringVar**
- **mainloop**

**Next:** 60. Qt Desktop Apps (`pyside6 (PyQt)`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/60_pyqt_pyside.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
# Tkinter comes with Python (Linux: sudo apt install python3-tk)
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.
4. **Run the app:** download `shop_logic.py`, `tk_shop.py` (Colab: Files panel on the left → right-click → Download) into the `python-lab` folder, then run `python tk_shop.py` in the same terminal.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.